In [ ]:
# %% Deep learning - Section 19.179
#    Code challenge 28: how wide the fully connected layer ?
#
#    1) Start from code from video 19.176 (gaussian synthetic dataset)
#    2) Add a new fully connected layer
#    3) Set the number of units in fc1 and fc2 to be {2x,2}
#    4) Test performance for x = {5,500}
#    5) Run the model on GPU
#    6) Plot final model losses and accuracies

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.


In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [12]:
# %% Generate data

# 2D Gaussian params
n_per_class = 1000
n_classes   = 2
img_size    = 91
x           = np.linspace(-4,4,img_size)
X,Y         = np.meshgrid(x,x)

widths      = [1.8,2.4]

# Preallocate tensors for images (N,channels,size,size) and labels (N)
images  = torch.zeros( n_classes*n_per_class,1,img_size,img_size )
labels  = torch.zeros( n_classes*n_per_class )

# Generate images
for i in range(n_classes*n_per_class):

    # Gaussian with random center offset (remainder trick for width, all even
    # images go into category 0, all odd images go into category 1)
    c = 2*np.random.randn(2)
    G = np.exp( -((X-c[0])**2 + (Y-c[1])**2 ) / (2*widths[i%2]**2) )

    # Layer some noise
    G = G + np.random.randn(img_size,img_size)/5

    # Add to tensor
    images[i,:,:,:] = torch.tensor(G).view(1,img_size,img_size)
    labels[i]       = i%2

labels = labels[:,None]


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5))/2
fig,axs = plt.subplots(3,7,figsize=(phi*7,7))

for i,ax in enumerate(axs.flatten()):

    pic = np.random.randint(2*n_per_class)
    G   = np.squeeze( images[pic,:,:] )

    ax.imshow(G,vmin=-1,vmax=1,cmap='jet')
    ax.set_title('Class %s'%int(labels[pic].item()))
    ax.set_xticks([])
    ax.set_yticks([])

plt.savefig('figure45_code_challenge_28.png')
plt.show()
files.download('figure45_code_challenge_28.png')


In [13]:
# %% Create train and test datasets

# Split data with scikitlearn (10% test data)
train_data,test_data,train_labels,test_labels = train_test_split(images,labels,test_size=0.1)

# Convert to PyTorch datasets
train_data = TensorDataset(train_data,train_labels)
test_data  = TensorDataset(test_data,test_labels)

# Convert into DataLoader objects
batch_size   = 32
train_loader = DataLoader(train_data,batch_size=batch_size,shuffle=True,drop_last=True)
test_loader  = DataLoader(test_data,batch_size=test_data.tensors[0].shape[0])


In [20]:
# %% Function to generate the model

# Combine custom class and nn.Sequential
def gen_model(fc_units):

    class Gauss_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Layers with nn.Sequential (activation function is treated as layer)
            self.model = nn.Sequential( nn.Conv2d(1,6,3,padding=1),     # out size: (91+2*1-3)/1 + 1 = 91
                                        nn.ReLU(),
                                        nn.AvgPool2d(2,2),              # out size: 91/2 = 45
                                        nn.Conv2d(6,4,3,padding=1),     # out size: (45+2*1-3)/1 + 1 = 45
                                        nn.ReLU(),
                                        nn.AvgPool2d(2,2),              # out size: 45/2 = 22
                                        nn.Flatten(),                   # vectorise
                                        nn.Linear(22*22*4,2*fc_units),
                                        nn.ReLU(),
                                        nn.Linear(2*fc_units,fc_units),
                                        nn.Linear(fc_units,1)           # out size: 1
                                        )

        def forward(self,x):
            return self.model(x)

    # Create model instance
    CNN = Gauss_CNN()

    # Loss function
    loss_fun = nn.BCEWithLogitsLoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [ ]:
# %% Test the model on one batch

fc_units = 100
CNN,loss_fun,optimizer = gen_model(fc_units)

X,y  = next(iter(train_loader))
yHat = CNN(X)

# Check sizes of output and target variable
print()
print(yHat.shape), print()
print(y.shape), print()

# Check loss
loss = loss_fun(yHat,y)
print(loss)


In [24]:
# %% Function to train the model

def train_model(fc_units):

    # Parameters, model instance, inizialise vars
    num_epochs = 10
    CNN,loss_fun,optimizer = gen_model(fc_units)

    # Ship model to GPU
    CNN.to(device)

    train_losses = []
    test_losses  = []
    train_acc    = []
    test_acc     = []

    # Loop over epochs
    for epoch_i in range(num_epochs):

        # Loop over training batches
        batch_acc  = []
        batch_loss = []

        for X,y in train_loader:

            # Ship data to GPU
            X = X.to(device)
            y = y.to(device)

            # Forward propagation and loss
            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Loss and accuracy from this batch
            yHat = yHat.cpu()
            y    = y.cpu()

            batch_loss.append(loss.item())
            batch_acc.append( torch.mean( ((yHat>0)==y).float() ).item() )

        train_losses.append( np.mean(batch_loss) )
        train_acc.append( 100*np.mean(batch_acc) )

        # Test accuracy
        CNN.eval()

        with torch.no_grad():
            X,y = next(iter(test_loader))

            # Ship to GPU (and y for loss calculation)
            X = X.to(device)
            y = y.to(device)

            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            yHat = yHat.cpu()
            y    = y.cpu()

            test_acc.append( 100*torch.mean( ((yHat>0)==y).float() ).item() )
            test_losses.append(loss.item())

        CNN.train()

    return train_acc,test_acc,train_losses,test_losses,CNN


In [25]:
# %% Parametric experiment on FC units

# This cell takes only ~40 secs on GPU and ~15 mins on CPU
# specify number of hidden units
n_FC_units = np.floor(np.linspace(5,500,20))

# initialize results matrix
results = np.zeros((len(n_FC_units),4))

for i,n_units in enumerate(n_FC_units):
    train_acc,test_acc,train_losses,test_losses,CNN = train_model(int(n_units))

    final_train_acc  = np.mean(train_acc[-2:])
    final_test_acc   = np.mean(test_acc[-2:])
    final_train_loss = np.mean(train_losses[-2:])
    final_test_loss  = np.mean(test_losses[-2:])

    results[i,:] = [ final_train_loss,final_test_loss,final_train_acc,final_test_acc ]


In [ ]:
# %% Plotting
#    Note how extra units and layers are just gratuitous extra complexity you
#    don't actually need

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*5,5))

ax[0].plot(n_FC_units,results[:,:2],'s-')
ax[0].set_xlabel('Number of units in final linear layer')
ax[0].set_ylabel('Loss (MSE)')
ax[0].set_title('Final model loss')
ax[0].legend(['Train','Test'])

ax[1].plot(n_FC_units,results[:,2:],'s-')
ax[1].set_xlabel('Number of units in final linear layer')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title('Final model test accuracy')
ax[1].legend(['Train','Test'])

plt.savefig('figure46_code_challenge_28.png')
plt.show()
files.download('figure46_code_challenge_28.png')
